In [37]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd

DOMAIN = "https://www.nikkei.com/"
page_start = 1
page_end = 2
titles = []
body_texts = []
img_urls = []

for page in range(page_start, page_end+1):
    #htmlの取得
    url = "https://www.nikkei.com/business/net-media/?page={}".format(page)
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    #データー取得-タイトル
    links= soup.find_all("a", class_="fauxBlockLink_f6t5v6i")
    #１ページ目のトップ部分はクラスが違うため、別途取得
    if page == 1:
        link_page_one = soup.find_all("a", class_="fauxBlockLink_fzas0ps")
        links.insert(0, link_page_one[0])
    for link in links:
    #U3000除く
        titles.append(link.get_text().replace("\u3000", ""))
    #本文取得



    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    for link in links:
        href = link.get("href")

        if not href:
            body_texts.append("")
            continue

        body_link = urljoin(DOMAIN, href)

        body_response = requests.get(body_link, headers=headers)
        body_soup = BeautifulSoup(body_response.text, "html.parser")

        paragraphs = body_soup.find_all("p", class_="paragraph_p18mfke4")

        if not paragraphs:
            print("本文が見つかりません:", body_link)
            body_texts.append("")
            continue

        body_text = "\n".join(p.get_text(strip=True) for p in paragraphs)
        body_texts.append(body_text)
     #画像取得
    img_class = "image_i1obkbgm"
    img_tags = soup.find_all(class_=img_class)

    #img_urls = []
    # for img_tag in img_tags:
    #     img_url = img_tag.get("src")
    #     if not img_url:
    #       img_urls.append("")
    #       continue
    #     #if img_url:
    #       img_urls.append(img_url)
    for link in links:
        article = link.find_parent()
        img_tag = article.find("img") if article else None

        if img_tag and img_tag.get("src"):
            img_urls.append(img_tag.get("src"))
        else:
            img_urls.append("")
news_date = {
    'title': titles,
    'body_texts': body_texts,
    'img_urls': img_urls
}

print(len(titles))
print(len(body_texts))
print(len(img_urls))
df = pd.DataFrame.from_dict(news_date)


df.to_csv("nikkei_news2.csv", index=None, encoding="utf-8-sig")






本文が見つかりません: https://www.nikkei.com/article/DGXZQOCD2826H0Y6A420C2000000/
本文が見つかりません: https://www.nikkei.com/article/DGXZQOUC152IJ0V10C26A5000000/
本文が見つかりません: https://www.nikkei.com/article/DGXZQOUC153EA0V10C26A5000000/
本文が見つかりません: https://www.nikkei.com/article/DGXZQOUC152CX0V10C26A5000000/
本文が見つかりません: https://www.nikkei.com/article/DGXZQOUC153JW0V10C26A5000000/
本文が見つかりません: https://www.nikkei.com/article/DGXZQOUC151H40V10C26A5000000/
本文が見つかりません: https://www.nikkei.com/article/DGXZQOUC151GS0V10C26A5000000/
本文が見つかりません: https://www.nikkei.com/article/DGXZQOCC12B4O0S6A510C2000000/
本文が見つかりません: https://www.nikkei.com/article/DGXZQOUC148VD0U6A510C2000000/
40
40
40


In [23]:
df.to_csv("nikkei_news2.csv", index=None, encoding="utf-8-sig")

In [26]:
#画像保存
for index, row in df.iterrows():
    image_url = row["img_urls"]
    filename = str(index)
    image = requests.get(image_url)
    with open("img/"+filename+".jpg", "wb") as f:
            f.write(image.content)

MissingSchema: Invalid URL '/.resources/k-components/icon/locked_square.rev-5fc2936.svg': No scheme supplied. Perhaps you meant https:///.resources/k-components/icon/locked_square.rev-5fc2936.svg?

 ポイントは、if not img_url では「srcがない画像タグ」しか処理できないことです。
  今回の問題は、img_tag は存在するけど、それが記事画像ではなく鍵アイコンだった、という点です


原因は、img_urls に記事画像ではなく、サイトUI用のアイコンが入っていることです。
  requests.get() 側の問題ではなく、画像URLの取得段階で違う画像を拾っています。

  特にこれは記事画像ではありません。

  /.resources/k-components/icon/locked_square.rev-5fc2936.svg

  日経の「鍵アイコン」です。
